In [ ]:
import os, re, time, glob, unicodedata
import numpy as np
import pandas as pd
import requests
from pybaseball import (cache, bwar_bat, bwar_pitch, chadwick_register,
                        statcast_batter_percentile_ranks,
                        statcast_pitcher_percentile_ranks)
cache.enable()


In [ ]:
from pybaseball import bwar_bat, bwar_pitch, cache

cache.enable()

bat = bwar_bat()
print(bat.shape)
print(bat.columns.tolist())
bat[bat["year_ID"] == 2023].head()

In [ ]:
import pandas as pd

In [ ]:
print(bat["year_ID"].min(), bat["year_ID"].max())

pit = bwar_pitch()
print(pit.shape)
print(pit.columns.tolist())
pit[pit["year_ID"] == 2023].head()

In [ ]:
bat_s = (bat[bat["PA"] > 0]
         .groupby(["mlb_ID", "year_ID"])
         .agg(name=("name_common", "first"), PA=("PA", "sum"), WAR=("WAR", "sum"))
         .reset_index())

pit_s = (pit.groupby(["mlb_ID", "year_ID"])
         .agg(name=("name_common", "first"), G=("G", "sum"), GS=("GS", "sum"),
              BIP=("BIP", "sum"), WAR=("WAR", "sum"))
         .reset_index())

print(bat_s[bat_s["year_ID"] >= 2022].groupby("year_ID")["PA"].sum())
print(bat_s[bat_s["year_ID"] == 2026].nlargest(5, "WAR")[["name", "PA", "WAR"]])

In [ ]:
import pandas as pd
import unicodedata, re

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return {"michael fiers": "mike fiers", "steven pearce": "steve pearce"}.get(s, s)

bat_s["key"] = bat_s["name"].apply(norm)
pit_s["key"] = pit_s["name"].apply(norm)

c = pd.read_csv("../data/processed/contracts.csv")
c["key"] = c["player_name"].apply(norm)
c["stat_year"] = c["signing_year"] - 1

hit = c[c["player_type"] == "hitter"].merge(
    bat_s[["key", "year_ID", "mlb_ID", "WAR"]],
    left_on=["key", "stat_year"], right_on=["key", "year_ID"], how="left")
pitch = c[c["player_type"] == "pitcher"].merge(
    pit_s[["key", "year_ID", "mlb_ID", "WAR"]],
    left_on=["key", "stat_year"], right_on=["key", "year_ID"], how="left")

matched = pd.concat([hit, pitch], ignore_index=True)

print("Contracts:", len(c), "| Rows after join:", len(matched))

# Drop contracts where one name matched two different players
amb = matched.duplicated(["player_name", "signing_year"], keep=False)
print("Excluding", amb.sum(), "rows from",
      matched[amb][["player_name", "signing_year"]].drop_duplicates().shape[0],
      "ambiguous contracts")
matched = matched[~amb].copy()

print(matched.shape)
print(matched.groupby("player_type")["mlb_ID"].apply(lambda s: s.notna().mean()))

matched.to_csv("../data/processed/contracts_matched.csv", index=False)

In [ ]:
seasons = {
    "hitter": bat_s[["mlb_ID", "year_ID", "key", "WAR"]],
    "pitcher": pit_s[["mlb_ID", "year_ID", "key", "WAR"]],
}

# 1. Fill in missing IDs by looking back up to 2 extra seasons
def find_id(row):
    if pd.notna(row["mlb_ID"]):
        return row["mlb_ID"]
    s = seasons[row["player_type"]]
    cand = s[(s["key"] == row["key"]) &
             (s["year_ID"].between(row["stat_year"] - 2, row["stat_year"]))]
    ids = cand["mlb_ID"].unique()
    return ids[0] if len(ids) == 1 else float("nan")

m = matched.copy()
m["mlb_ID"] = m.apply(find_id, axis=1)
print("Share with a player ID:", round(m["mlb_ID"].notna().mean(), 3))

# 2. Look up WAR for each of the three seasons before signing
lookup = {}
for t, s in seasons.items():
    for mid, yr, w in zip(s["mlb_ID"], s["year_ID"], s["WAR"]):
        lookup[(t, mid, yr)] = w

for back, col in enumerate(["war_1", "war_2", "war_3"]):
    m[col] = [
        lookup.get((t, i, y - back), 0.0) if pd.notna(i) else float("nan")
        for t, i, y in zip(m["player_type"], m["mlb_ID"], m["stat_year"])
    ]

# 3. Weighted average (most recent season counts most)
m["war_w"] = (5 * m["war_1"] + 4 * m["war_2"] + 3 * m["war_3"]) / 12

m.to_csv("../data/processed/contracts_with_war.csv", index=False)
print(m.shape)
print(m[["war_w", "war_1", "aav"]].corr()["aav"])
m.nlargest(8, "aav")[["player_name", "signing_year", "war_3", "war_2", "war_1", "war_w", "aav"]]

In [ ]:
from pybaseball import chadwick_register

reg = chadwick_register()
print(reg.shape)
print([c for c in reg.columns if "birth" in c.lower() or "mlbam" in c.lower()])
reg.head(3)

In [ ]:
import requests, time

ids = sorted(int(i) for i in m["mlb_ID"].dropna().unique())
print("Players to look up:", len(ids))

births = {}
for i in range(0, len(ids), 100):
    batch = ids[i:i+100]
    r = requests.get(
        "https://statsapi.mlb.com/api/v1/people",
        params={"personIds": ",".join(map(str, batch))},
        timeout=30,
    )
    r.raise_for_status()
    for p in r.json().get("people", []):
        if "birthDate" in p:
            births[p["id"]] = p["birthDate"]
    time.sleep(0.5)

print("Birth dates found:", len(births))

m["birth_date"] = pd.to_datetime(m["mlb_ID"].map(births))

# Age at signing: the offseason starts after season N, so use Dec 1 of the stat year
signing_date = pd.to_datetime(m["stat_year"].astype(int).astype(str) + "-12-01")
m["age_at_signing"] = ((signing_date - m["birth_date"]).dt.days / 365.25).round(1)

print(m["age_at_signing"].describe())
m.to_csv("../data/processed/contracts_with_age.csv", index=False)
m.nlargest(5, "aav")[["player_name", "signing_year", "birth_date", "age_at_signing", "aav"]]

In [ ]:
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks

sav_b = statcast_batter_percentile_ranks(2023)
print(sav_b.shape)
print(sav_b.columns.tolist())

sav_p = statcast_pitcher_percentile_ranks(2023)
print(sav_p.shape)
print(sav_p.columns.tolist())

In [ ]:
import pandas as pd, time
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks, cache

cache.enable()

hit_frames, pit_frames = [], []
for yr in range(2015, 2027):
    try:
        hb = statcast_batter_percentile_ranks(yr)
        hb["year"] = yr
        pp = statcast_pitcher_percentile_ranks(yr)
        pp["year"] = yr
        hit_frames.append(hb)
        pit_frames.append(pp)
        print(yr, "hitters:", len(hb), "pitchers:", len(pp))
    except Exception as e:
        print(yr, "FAILED:", str(e)[:100])
    time.sleep(1)

sav_hit = pd.concat(hit_frames, ignore_index=True)
sav_pit = pd.concat(pit_frames, ignore_index=True)
sav_hit.to_csv("../data/processed/savant_hitters.csv", index=False)
sav_pit.to_csv("../data/processed/savant_pitchers.csv", index=False)

# Are these 0-100 percentiles?
print(sav_hit[["xwoba", "exit_velocity"]].describe().loc[["min", "max"]])

# Share of players with a value for each metric, by year
cols = ["xwoba", "brl_percent", "exit_velocity", "chase_percent", "sprint_speed", "bat_speed"]
print(sav_hit.groupby("year")[cols].apply(lambda g: g.notna().mean()).round(2))

# Does player_id match the MLB IDs we already have?
d = pd.read_csv("../data/processed/contracts_with_age.csv")
ids = set(d["mlb_ID"].dropna().astype(int))
found = ids & (set(sav_hit["player_id"].astype(int)) | set(sav_pit["player_id"].astype(int)))
print("Contract players found in Savant data:", round(len(found) / len(ids), 3))

In [ ]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")
sub = c[(c["player_type"] == "hitter") & (c["signing_year"] >= 2025)
        & c["war_w"].between(1.3, 2.3) & (c["total_value"] >= 5_000_000)].copy()
sub["aav_m"] = (sub["aav"] / 1e6).round(1)

print("Comparable hitters:", len(sub))
print("Median AAV: $%.1fM" % sub["aav_m"].median())
print("Middle half: $%.1fM to $%.1fM" % (sub["aav_m"].quantile(.25), sub["aav_m"].quantile(.75)))
sub.sort_values("aav_m", ascending=False)[
    ["player_name", "signing_year", "age_at_signing", "war_w", "aav_m"]].head(15)

In [ ]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")
qo1 = c[(c["qualifying_offer"] == 1) & (c["years"] == 1)].copy()
qo1["aav_m"] = (qo1["aav"] / 1e6).round(2)

print("One-year deals with a qualifying offer flag, by class:")
print(qo1.groupby("signing_year")["aav_m"].agg(["count", "min", "max"]))
print()
print(qo1[qo1["signing_year"] >= 2024]
      .sort_values(["signing_year", "aav_m"], ascending=[True, False])
      [["player_name", "signing_year", "war_w", "aav_m"]].to_string(index=False))

In [ ]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")

# League-set qualifying offer amounts ($M) by signing class.
# 2025 and 2026 are confirmed by your own output; the earlier years are from my memory,
# so check the printout below: each year's dropped players should share one salary.
qo_amount = {2016: 15.8, 2017: 17.2, 2018: 17.4, 2019: 17.9, 2020: 17.8,
             2021: 18.9, 2022: 18.4, 2023: 19.65, 2024: 20.3, 2025: 21.05, 2026: 22.02}

aav_m = c["aav"] / 1e6
amt = c["signing_year"].map(qo_amount)
accepted = (c["qualifying_offer"] == 1) & (c["years"] == 1) & ((aav_m - amt).abs() < 0.06)

print("Dropping", accepted.sum(), "qualifying-offer acceptances:")
print(c[accepted].assign(aav_m=aav_m[accepted].round(2))
      .sort_values("signing_year")[["player_name", "signing_year", "aav_m"]].to_string(index=False))

c[~accepted].to_csv("../data/processed/contracts_model.csv", index=False)
print("Saved contracts_model.csv with", (~accepted).sum(), "contracts")

In [ ]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")

# League-set qualifying offer amounts ($M) by signing class.
# 2025 and 2026 are confirmed by your own output; the earlier years are from memory.
qo_amount = {2016: 15.8, 2017: 17.2, 2018: 17.4, 2019: 17.9, 2020: 17.8,
             2021: 18.9, 2022: 18.4, 2023: 19.65, 2024: 20.3, 2025: 21.05, 2026: 22.02}

aav_m = c["aav"] / 1e6
amt = c["signing_year"].map(qo_amount)
one_year_qo = (c["qualifying_offer"] == 1) & (c["years"] == 1)
accepted = one_year_qo & ((aav_m - amt).abs() < 0.06)

print("Dropping", accepted.sum(), "qualifying-offer acceptances:")
print(c[accepted].assign(aav_m=aav_m[accepted].round(2))
      .sort_values("signing_year")[["player_name", "signing_year", "aav_m"]].to_string(index=False))

kept = one_year_qo & ~accepted
print("\nOne-year deals with a QO flag that were KEPT (check these):")
print(c[kept].assign(aav_m=aav_m[kept].round(2))
      .sort_values("signing_year")[["player_name", "signing_year", "aav_m"]].to_string(index=False))

c[~accepted].to_csv("../data/processed/contracts_model.csv", index=False)
print("\nSaved contracts_model.csv with", (~accepted).sum(), "contracts")

In [ ]:
import pandas as pd

raw = pd.read_csv("../data/processed/contracts_2024.csv")
pcol = [c for c in raw.columns if c.startswith("Player")][0]
print(raw[raw[pcol].astype(str).str.contains("Belt|Junis|Urshela|Heyward", case=False)])